# Notebook 4 (V9) — Bubble-Existence Diagnostics (Theorem 1)

V9 Theorem 1 (`thm_bubble_characterization`): if $z_0 = u$, there is a bubble in the per-variety US stock at date 0 if and only if
$$
(1a) \quad \sum_{t=1}^{\infty} \frac{d_t^u}{q_t^u} < \infty,
\qquad
(1b) \quad \sum_{t=1}^{\infty} \frac{q_t^b + d_t^b}{q_t^u}\left(\frac{C_t^u}{C_t^b}\right)^\gamma < \infty.
$$

**Production-side mechanism (Theorem `thm_prod_sufficient`):**
Under V9 Assumptions (CES production, $\rho_{US}>1$, $\xi_u>\nu_u>\nu_b$, $\gamma<1$), 
- $\varphi_{US,t}^u \asymp N_{US,t}^{-\psi_{US}/\rho_{US}}$ ⇒ (1a) holds (geometric);
- $q_t^u \asymp N_{US,t}^{\nu_u-1}$, $q_t^b+d_t^b \asymp N_{US,t}^{\nu_b-1}$, $C_t^z \asymp N_{US,t}^{\nu_z}$ ⇒ (1b) reduces to $N_{US,t}^{(\nu_b-\nu_u)(1-\gamma)}$, which sums when $\nu_u>\nu_b$ and $\gamma<1$.

Hence under those primitives, the per-variety US stock contains a bubble at date 0.

In [ ]:
using Pkg; Pkg.activate(".")
include("TwoCountryProductionOLG.jl")
using Plots, LaTeXStrings, Printf
gr()

In [ ]:
p = ProductionParams(T_max=30, common_world_growth=true, branch_iters=20)
result = run_production_simulation(p; verbose=true)

## 1. Theorem 1 Conditions

In [ ]:
diag = result.diagnostics
T = p.T_max
tt = 1:T

println("═══ V9 THEOREM 1 DIAGNOSTICS ═══")
@printf("  Cond (1a) Σ d^u/q^u   = %.6f  (converges: %s)\n",
        diag.cum_1a[end], diag.cond_1a_converges)
@printf("  Cond (1b) Σ branch    = %.6f  (converges: %s)\n",
        diag.cum_1b[end], diag.cond_1b_converges)
@printf("  Total leakage Σ a_t   = %.6f\n", diag.cum_a_t[end])
@printf("  BUBBLE EXISTS         : %s\n", diag.bubble_exists)

## 2. Per-Period Decay Rates

In [ ]:
p1 = plot(tt, diag.cond_1a, lw=2, marker=:circle, label=L"d_t^u/q_t^u",
          xlabel="period t", ylabel=L"d^u/q^u", yscale=:log10,
          title="Theorem 1 cond (1a): per-variety dividend yield")

p2 = plot(tt, diag.cond_1b, lw=2, marker=:circle,
          label=L"\frac{q_t^b+d_t^b}{q_t^u}(\frac{C_t^u}{C_t^b})^{\gamma}",
          xlabel="period t", ylabel="branch term", yscale=:log10,
          title="Theorem 1 cond (1b): switch-branch term")

p3 = plot(tt, diag.a_t, lw=2, marker=:circle, label=L"a_t",
          xlabel="period t", ylabel=L"a_t (\mathrm{leakage \,ratio})", yscale=:log10,
          title="Bubble-recursion leakage ratio")

p4 = plot(tt, diag.cum_1a, lw=2, marker=:circle, label="cumulative (1a)",
          xlabel="period t", ylabel="cumulative",
          title="Cumulative summability terms")
plot!(p4, tt, diag.cum_1b, lw=2, marker=:square, label="cumulative (1b)")
plot!(p4, tt, diag.cum_a_t, lw=2, marker=:utriangle, label=L"\Sigma a_t")

plot(p1, p2, p3, p4, layout=(2,2), size=(1000, 750))

## 3. φ-Decay vs. Predicted Asymptote

By Lemma `lem_prod_orders`, $\varphi_{US,t}^u \asymp N_{US,t}^{-\psi_{US}/\rho_{US}}$.
Verify the empirical decay rate from the numerical solution.

In [ ]:
φ_US = [result.u_path[t].φ_US for t in tt]
N_US = [result.u_path[t].N_US for t in tt]

ψ_US = (p.ξ_u - p.ν_u) * (p.ρ_US - 1)
decay_pred = ψ_US / p.ρ_US
@printf("V9 predicted decay exponent: -ψ_US/ρ_US = -%.4f\n", decay_pred)

# Empirical decay rate via log-log regression
logN = log.(N_US[2:end] ./ N_US[1])
logφ = log.(φ_US[2:end] ./ φ_US[1])
slope_emp = (sum(logN .* logφ) / sum(logN.^2))   # zero-intercept fit
@printf("Empirical slope of log(φ_US) vs log(N_US): %.4f\n", slope_emp)
@printf("Predicted slope:                          -%.4f\n", decay_pred)

scatter(N_US, φ_US, label="empirical", xscale=:log10, yscale=:log10,
        xlabel=L"N_{US,t}", ylabel=L"\varphi_{US,t}^u",
        title="V9 production-side decay rate")
plot!(N_US, φ_US[1] .* (N_US ./ N_US[1]).^(-decay_pred), lw=2,
      label="V9 asymptote slope -$(round(decay_pred, digits=3))")

## 4. Per-Variety vs. Aggregate Price-Dividend Ratios

V9 distinguishes per-variety and aggregate objects. Per-variety ratio:
$$
\frac{q_{US,t}^u}{d_{US,t}^u} = \frac{1}{a_{US}\frac{1-\vartheta_{US}}{\vartheta_{US}}\varphi_{US,t}^u H_{US}} \to \infty.
$$
Aggregate ratio (using $\mathcal{D}_{i,t} = N_{i,t}d_{i,t}$ vs. $\mathcal{Q}_{i,t} = N_{i,t+1}q_{i,t}$):
$$
\frac{\mathcal{Q}_{US,t}}{\mathcal{D}_{US,t}} = \frac{N_{US,t+1}}{N_{US,t}}\cdot\frac{q_{US,t}^u}{d_{US,t}^u} = G_{N,US,t}\cdot\frac{q_{US,t}^u}{d_{US,t}^u},
$$
so the two have the same asymptotic order whenever the knowledge growth factor is bounded.

In [ ]:
q_US = [result.u_path[t].q_US for t in tt]
d_US = [result.u_path[t].d_US for t in tt]
Q_US = [result.u_path[t].Q_US for t in tt]
D_US = [N_US[t] * d_US[t] for t in tt]
qd_per   = q_US ./ d_US
QD_agg   = Q_US ./ D_US

p1 = plot(tt, qd_per, lw=2, marker=:circle, label=L"q_{US}^u / d_{US}^u",
          xlabel="period t", ylabel="price-dividend ratio",
          title="Per-variety US price-dividend ratio (rises ⇒ bubble)")

p2 = plot(tt, QD_agg, lw=2, marker=:square, label=L"\mathcal{Q}_{US}/\mathcal{D}_{US}",
          xlabel="period t", ylabel="aggregate ratio",
          title="Aggregate US price-dividend ratio")

plot(p1, p2, layout=(1,2), size=(1000, 380))

## 5. Common-World-Growth Diagnostic

In [ ]:
rel_size = [result.u_path[t].Y_W / result.u_path[t].Y_US for t in tt]
p1 = plot(tt, rel_size, lw=2, marker=:circle, label=L"Y_W/Y_{US}",
          xlabel="period t", ylabel="relative size",
          title="Relative country size along u-branch")

# Common-world-growth gap on the absorbing branch
gap = [result.bgp_seq[t].G_N_US^p.ν_b - result.bgp_seq[t].G_N_W^p.ξ_W for t in 1:T+1]
p2 = plot(0:T, gap, lw=2, marker=:circle, label=L"G_b - G_W",
          xlabel="BGP index", ylabel="gap",
          title="Common-growth gap along absorbing-branch BGPs")
hline!(p2, [0.0], ls=:dash, color=:black, label="")

plot(p1, p2, layout=(1,2), size=(1000, 380))

## Summary

- **Per-period dividend yield** $d_t^u/q_t^u$ falls geometrically — confirming Theorem 1 condition (1a).
- **Switch-branch term** $\frac{q_t^b+d_t^b}{q_t^u}(C_t^u/C_t^b)^\gamma$ falls at rate $N_{US,t}^{(\nu_b-\nu_u)(1-\gamma)}$ — Theorem 1 condition (1b).
- **Empirical $\varphi$-decay** matches the V9 asymptote $N_{US,t}^{-\psi_{US}/\rho_{US}}$.
- **Per-variety price-dividend ratio** rises monotonically: this is the production-side bubble mechanism.
- Under common-world-growth calibration, relative country size stays bounded.

Next notebook: simulate a sample regime path with switch date and produce the V9 transition figures.